In [0]:
%sql
use `for-job-prac-lms`.default

In [0]:
# Incremental Ingestion using Auto Loader
from pyspark.sql.functions import *

print("Starting incremental ingestion...")


raw_path = "abfss://letssayapi@jobdata.dfs.core.windows.net/letsayapi/"

# CHECKPOINT LOCATION

checkpoint_path = "abfss://letssayapi@jobdata.dfs.core.windows.net/checkpoints/ramen_ingestion"


schema_path = "abfss://letssayapi@jobdata.dfs.core.windows.net/schema/ramen_ingestion"


# READ NEW FILES USING AUTO LOADER

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(raw_path)
)

# CLEAN COLUMN NAMES

clean_columns = []

for col_name in stream_df.columns:
    cleaned = (
        col_name.strip()
        .lower()
        .replace(" ", "_")
        .replace("#", "num")
    )
    clean_columns.append(cleaned)

stream_df = stream_df.toDF(*clean_columns)

# WRITE TO BRONZE TABLE


query = (
    stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("`for-job-prac-lms`.default.bronze_ramen_reviews")
)

query.awaitTermination()

print("Incremental ingestion completed successfully")

Starting incremental ingestion...
Incremental ingestion completed successfully
